In [12]:
import pandas as pd
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


In [13]:
df = pd.read_csv('sales_data_advanced.csv', parse_dates=['Date'], dayfirst=True)

In [14]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

In [15]:
df_filled = df.fillna(0)

In [16]:
print(df.isnull().sum())

Date                0
Product_ID          0
Product_Name        0
Quantity sold       0
GDP                 0
Inflation           0
Campaign_Active     0
Promotion_Active    0
Holiday             0
Special_Event       0
Temperature         0
Precipitation       0
dtype: int64


In [17]:
df = df.dropna()
print(df.isnull().sum())

Date                0
Product_ID          0
Product_Name        0
Quantity sold       0
GDP                 0
Inflation           0
Campaign_Active     0
Promotion_Active    0
Holiday             0
Special_Event       0
Temperature         0
Precipitation       0
dtype: int64


In [18]:
print(df.dtypes)

Date                datetime64[ns]
Product_ID                   int64
Product_Name                object
Quantity sold                int64
GDP                        float64
Inflation                  float64
Campaign_Active              int64
Promotion_Active             int64
Holiday                      int64
Special_Event                int64
Temperature                float64
Precipitation              float64
dtype: object


In [19]:
# Display summary statistics
print(df.describe())


                                Date   Product_ID  Quantity sold          GDP  \
count                           3930  3930.000000    3930.000000  3930.000000   
mean   2023-07-15 23:59:59.999999744     5.500000      54.177099  1484.622775   
min              2023-01-01 00:00:00     1.000000      10.000000  1001.149358   
25%              2023-04-09 00:00:00     3.000000      31.000000  1243.033384   
50%              2023-07-16 00:00:00     5.500000      54.000000  1468.091022   
75%              2023-10-22 00:00:00     8.000000      77.000000  1745.902302   
max              2024-01-28 00:00:00    10.000000      99.000000  1996.335270   
std                              NaN     2.872647      26.220722   285.228180   

         Inflation  Campaign_Active  Promotion_Active      Holiday  \
count  3930.000000      3930.000000       3930.000000  3930.000000   
mean      3.054678         0.529262          0.544529     0.002545   
min       1.015710         0.000000          0.000000     0.

In [20]:
df_filled['Day_of_Week'] = df_filled['Date'].dt.dayofweek
df_filled['Month_of_Year'] = df_filled['Date'].dt.month
preprocessor = ColumnTransformer(
    transformers=[('encoder', OneHotEncoder(), ['Product_Name'])],
    remainder='passthrough'
)
X = preprocessor.fit_transform(df_filled.drop(['Quantity sold', 'Date'], axis=1))
y = df_filled['Quantity sold']


In [21]:
# Check if 'Product_ID' is present in your dataset
if 'Product_ID' in df_filled.columns:
    print("Product_ID is present in the dataset.")
else:
    print("Product_ID is missing from the dataset.")


Product_ID is present in the dataset.


In [22]:
# Apply one-hot encoding to the 'Product_Name' and 'Product_ID' columns
preprocessor = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(), ['Product_Name', 'Product_ID']),
        ('numeric', 'passthrough', ['Day_of_Week', 'Month_of_Year', 'Temperature', 'Precipitation', 'Inflation', 'Holiday', 'Promotion_Active', 'GDP', 'Campaign_Active', 'Special_Event'])
    ],
    remainder='drop'
)


In [23]:

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X, y)


RandomForestRegressor(random_state=42)

In [24]:
# Transform the features
X = preprocessor.fit_transform(df_filled.drop(['Quantity sold', 'Date'], axis=1))
y = df_filled['Quantity sold']

# Create a Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X, y)

# Get the current date
current_date = datetime.now().date()
print(current_date)

2025-02-15


In [25]:


# Forecast for the next one month for all products
product_ids = df_filled['Product_ID'].unique()
for product_id in product_ids:
    # Adjust X_future with relevant feature values for the future period
    X_future = preprocessor.transform(pd.DataFrame({
        'Product_Name': [df_filled.loc[df_filled['Product_ID'] == product_id, 'Product_Name'].iloc[0]],  # Specify the product for forecasting
        'Product_ID': [product_id],  # Use the existing Product_ID
        'Day_of_Week': [current_date.weekday()],  # Use the current day of the week
        'Month_of_Year': [current_date.month],  # Use the current month
        'Temperature': [25.0],  # Placeholder value for temperature
        'Precipitation': [0.0],  # Placeholder value for precipitation
        'Inflation': [2.5],  # Placeholder value for inflation
        'Holiday': [0],  # Placeholder value for holiday (0 for no holiday, 1 for holiday)
        'Promotion_Active': [1],  # Placeholder value for promotion status (0 for inactive, 1 for active)
        'GDP': [2000.0],  # Placeholder value for GDP
        'Campaign_Active': [0],  # Placeholder value for campaign status (0 for inactive, 1 for active)
        'Special_Event': [0]  # Placeholder value for special event status (0 for no event, 1 for event)
    }))

    # Make predictions for the next one month
    future_date = current_date + timedelta(days=30)
    future_date_features = pd.DataFrame({
        'Product_Name': [df_filled.loc[df_filled['Product_ID'] == product_id, 'Product_Name'].iloc[0]],  # Specify the product for forecasting
        'Product_ID': [product_id],  # Use the existing Product_ID
        'Day_of_Week': [future_date.weekday()],  # Adjust with future date
        'Month_of_Year': [future_date.month],  # Adjust with future date
        'Temperature': [50.0],  # Placeholder value for temperature
        'Precipitation': [0.0],  # Placeholder value for precipitation
        'Inflation': [4.5],  # Placeholder value for inflation
        'Holiday': [0],  # Placeholder value for holiday (0 for no holiday, 1 for holiday)
        'Promotion_Active': [1],  # Placeholder value for promotion status (0 for inactive, 1 for active)
        'GDP': [1015.165508],  # Placeholder value for GDP
        'Campaign_Active': [1],  # Placeholder value for campaign status (0 for inactive, 1 for active)
        'Special_Event': [0]  # Placeholder value for special event status (0 for no event, 1 for event)
    })
    X_future = preprocessor.transform(future_date_features)

    # Make predictions for the next one month
    forecasted_quantity = rf_model.predict(X_future)

    rounded_forecast = round(forecasted_quantity[0])

    # Print or use the rounded_forecast as needed
    print(f"Forecast for Product_ID {product_id} for the next one month: {rounded_forecast}")


Forecast for Product_ID 1 for the next one month: 63
Forecast for Product_ID 2 for the next one month: 57
Forecast for Product_ID 3 for the next one month: 55
Forecast for Product_ID 4 for the next one month: 47
Forecast for Product_ID 5 for the next one month: 49
Forecast for Product_ID 6 for the next one month: 59
Forecast for Product_ID 7 for the next one month: 59
Forecast for Product_ID 8 for the next one month: 47
Forecast for Product_ID 9 for the next one month: 49
Forecast for Product_ID 10 for the next one month: 53


In [26]:


# Forecast for the next one month for all products
product_ids = df_filled['Product_ID'].unique()
for product_id in product_ids:
    # Adjust X_future with relevant feature values for the future period
    X_future = preprocessor.transform(pd.DataFrame({
        'Product_Name': [df_filled.loc[df_filled['Product_ID'] == product_id, 'Product_Name'].iloc[0]],  # Specify the product for forecasting
        'Product_ID': [product_id],  # Use the existing Product_ID
        'Day_of_Week': [current_date.weekday()],  # Use the current day of the week
        'Month_of_Year': [current_date.month],  # Use the current month
        'Temperature': [25.0],  # Placeholder value for temperature
        'Precipitation': [0.0],  # Placeholder value for precipitation
        'Inflation': [2.5],  # Placeholder value for inflation
        'Holiday': [0],  # Placeholder value for holiday (0 for no holiday, 1 for holiday)
        'Promotion_Active': [1],  # Placeholder value for promotion status (0 for inactive, 1 for active)
        'GDP': [2000.0],  # Placeholder value for GDP
        'Campaign_Active': [0],  # Placeholder value for campaign status (0 for inactive, 1 for active)
        'Special_Event': [0]  # Placeholder value for special event status (0 for no event, 1 for event)
    }))

    # Make predictions for the next one month
    future_date = current_date + timedelta(days=30)
    future_date_features = pd.DataFrame({
        'Product_Name': [df_filled.loc[df_filled['Product_ID'] == product_id, 'Product_Name'].iloc[0]],  # Specify the product for forecasting
        'Product_ID': [product_id],  # Use the existing Product_ID
        'Day_of_Week': [future_date.weekday()],  # Adjust with future date
        'Month_of_Year': [future_date.month],  # Adjust with future date
        'Temperature': [25.0],  # Placeholder value for temperature
        'Precipitation': [0.0],  # Placeholder value for precipitation
        'Inflation': [2.5],  # Placeholder value for inflation
        'Holiday': [0],  # Placeholder value for holiday (0 for no holiday, 1 for holiday)
        'Promotion_Active': [1],  # Placeholder value for promotion status (0 for inactive, 1 for active)
        'GDP': [2000.0],  # Placeholder value for GDP
        'Campaign_Active': [0],  # Placeholder value for campaign status (0 for inactive, 1 for active)
        'Special_Event': [0]  # Placeholder value for special event status (0 for no event, 1 for event)
    })
    X_future = preprocessor.transform(future_date_features)

    # Make predictions for the next one month
    forecasted_quantity = rf_model.predict(X_future)

    # Print or use the forecasted_quantity as needed
    print(f"Forecast for Product_ID {product_id} for the next one month: {forecasted_quantity}")


Forecast for Product_ID 1 for the next one month: [66.51]
Forecast for Product_ID 2 for the next one month: [62.21]
Forecast for Product_ID 3 for the next one month: [68.48]
Forecast for Product_ID 4 for the next one month: [60.52]
Forecast for Product_ID 5 for the next one month: [57.09]
Forecast for Product_ID 6 for the next one month: [56.78]
Forecast for Product_ID 7 for the next one month: [68.3]
Forecast for Product_ID 8 for the next one month: [52.62]
Forecast for Product_ID 9 for the next one month: [57.16]
Forecast for Product_ID 10 for the next one month: [50.85]


In [27]:


# Forecast for the next one month for all products
product_ids = df_filled['Product_ID'].unique()
for product_id in product_ids:
    # Extract product name
    product_name = df_filled.loc[df_filled['Product_ID'] == product_id, 'Product_Name'].iloc[0]

    # Adjust X_future with relevant feature values for the future period
    X_future = preprocessor.transform(pd.DataFrame({
        'Product_Name': [product_name],  # Specify the product for forecasting
        'Product_ID': [product_id],  # Use the existing Product_ID
        'Day_of_Week': [current_date.weekday()],  # Use the current day of the week
        'Month_of_Year': [current_date.month],  # Use the current month
        'Temperature': [25.0],  # Placeholder value for temperature
        'Precipitation': [0.0],  # Placeholder value for precipitation
        'Inflation': [2.5],  # Placeholder value for inflation
        'Holiday': [0],  # Placeholder value for holiday (0 for no holiday, 1 for holiday)
        'Promotion_Active': [1],  # Placeholder value for promotion status (0 for inactive, 1 for active)
        'GDP': [2000.0],  # Placeholder value for GDP
        'Campaign_Active': [0],  # Placeholder value for campaign status (0 for inactive, 1 for active)
        'Special_Event': [0]  # Placeholder value for special event status (0 for no event, 1 for event)
    }))

    # Make predictions for the next one month
    future_date = current_date + timedelta(days=30)
    future_date_features = pd.DataFrame({
        'Product_Name': [product_name],  # Specify the product for forecasting
        'Product_ID': [product_id],  # Use the existing Product_ID
        'Day_of_Week': [future_date.weekday()],  # Adjust with future date
        'Month_of_Year': [future_date.month],  # Adjust with future date
        'Temperature': [25.0],  # Placeholder value for temperature
        'Precipitation': [0.0],  # Placeholder value for precipitation
        'Inflation': [2.5],  # Placeholder value for inflation
        'Holiday': [0],  # Placeholder value for holiday (0 for no holiday, 1 for holiday)
        'Promotion_Active': [1],  # Placeholder value for promotion status (0 for inactive, 1 for active)
        'GDP': [2000.0],  # Placeholder value for GDP
        'Campaign_Active': [0],  # Placeholder value for campaign status (0 for inactive, 1 for active)
        'Special_Event': [0]  # Placeholder value for special event status (0 for no event, 1 for event)
    })
    X_future = preprocessor.transform(future_date_features)

    # Make predictions for the next one month
    forecasted_quantity = rf_model.predict(X_future)
    rounded_forecast = round(forecasted_quantity[0])

    # Print or use the rounded_forecast as needed
    print(f"Estimated sales of the ({product_id}){product_name} for the next one month: {rounded_forecast}")


Estimated sales of the (1)Floral Sundress for the next one month: 67
Estimated sales of the (2)Evening Gown for the next one month: 62
Estimated sales of the (3)Business Casual Dress for the next one month: 68
Estimated sales of the (4)Sequin Party Dress for the next one month: 61
Estimated sales of the (5)Summer Maxi Dress for the next one month: 57
Estimated sales of the (6)Bridal Gown for the next one month: 57
Estimated sales of the (7)Traditional Ethnic Dress for the next one month: 68
Estimated sales of the (8)Maternity Nursing Dress for the next one month: 53
Estimated sales of the (9)Sporty Athleisure Dress for the next one month: 57
Estimated sales of the (10)Customizable Dress for the next one month: 51


In [28]:
# Forecast for the next one month for all products
forecast_data = []

for product_name in df_filled['Product_Name'].unique():
    # Adjust X_future with relevant feature values for the future period
    X_future = preprocessor.transform(pd.DataFrame({
        'Product_Name': [product_name],  # Specify the product for forecasting
        'Product_ID': [df_filled.loc[df_filled['Product_Name'] == product_name, 'Product_ID'].iloc[0]],  # Use the existing Product_ID
        'Day_of_Week': [current_date.weekday()],  # Use the current day of the week
        'Month_of_Year': [current_date.month],  # Use the current month
        'Temperature': [25.0],  # Placeholder value for temperature
        'Precipitation': [0.0],  # Placeholder value for precipitation
        'Inflation': [2.5],  # Placeholder value for inflation
        'Holiday': [0],  # Placeholder value for holiday (0 for no holiday, 1 for holiday)
        'Promotion_Active': [1],  # Placeholder value for promotion status (0 for inactive, 1 for active)
        'GDP': [2000.0],  # Placeholder value for GDP
        'Campaign_Active': [0],  # Placeholder value for campaign status (0 for inactive, 1 for active)
        'Special_Event': [0]  # Placeholder value for special event status (0 for no event, 1 for event)
    }))
    future_date = current_date + timedelta(days=30)
    future_date_features = pd.DataFrame({
        'Product_Name': [product_name],  # Specify the product for forecasting
        'Product_ID': [df_filled.loc[df_filled['Product_Name'] == product_name, 'Product_ID'].iloc[0]],  # Use the existing Product_ID
        'Day_of_Week': [future_date.weekday()],  # Adjust with future date
        'Month_of_Year': [future_date.month],  # Adjust with future date
        'Temperature': [25.0],  # Placeholder value for temperature
        'Precipitation': [0.0],  # Placeholder value for precipitation
        'Inflation': [2.5],  # Placeholder value for inflation
        'Holiday': [1],  # Placeholder value for holiday (0 for no holiday, 1 for holiday)
        'Promotion_Active': [1],  # Placeholder value for promotion status (0 for inactive, 1 for active)
        'GDP': [2000.0],  # Placeholder value for GDP
        'Campaign_Active': [1],  # Placeholder value for campaign status (0 for inactive, 1 for active)
        'Special_Event': [0]  # Placeholder value for special event status (0 for no event, 1 for event)
    })
    X_future = preprocessor.transform(future_date_features)
    forecasted_quantity = rf_model.predict(X_future)
    forecast_data.append({
        'Product_Name': product_name,
        'Forecasted_Quantity': round(forecasted_quantity[0])
    })
sorted_forecast_data = sorted(forecast_data, key=lambda x: x['Forecasted_Quantity'], reverse=True)
for i, forecast in enumerate(sorted_forecast_data[:10]):
    print(f"Rank {i+1}: {forecast['Product_Name']} - Forecasted Quantity: {forecast['Forecasted_Quantity']}")


Rank 1: Business Casual Dress - Forecasted Quantity: 69
Rank 2: Traditional Ethnic Dress - Forecasted Quantity: 68
Rank 3: Floral Sundress - Forecasted Quantity: 67
Rank 4: Evening Gown - Forecasted Quantity: 65
Rank 5: Sequin Party Dress - Forecasted Quantity: 63
Rank 6: Summer Maxi Dress - Forecasted Quantity: 58
Rank 7: Sporty Athleisure Dress - Forecasted Quantity: 57
Rank 8: Bridal Gown - Forecasted Quantity: 56
Rank 9: Customizable Dress - Forecasted Quantity: 51
Rank 10: Maternity Nursing Dress - Forecasted Quantity: 49
